In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from scipy.stats import entropy, iqr

In [15]:
df = pd.read_csv("../data/ble_data_labeled_cleaned.csv")
df.head()

,user_id,timestamp,mac_address,RSSI,power,location
0,90,2023-04-10 14:21:46.003,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
1,90,2023-04-10 14:21:46.008,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
2,90,2023-04-10 14:21:46.012,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
3,90,2023-04-10 14:21:46.018,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
4,90,2023-04-10 14:21:46.024,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen


In [16]:
def window_ble_data(df, window_size_s=10.0, step_size_s=5.0):
    df = df.sort_values("timestamp").copy()

    if not np.issubdtype(df["timestamp"].dtype, np.datetime64):
        df["timestamp"] = pd.to_datetime(df["timestamp"])

    start = df["timestamp"].min()
    end   = df["timestamp"].max()

    windows = []
    t = start

    while t + pd.Timedelta(seconds=window_size_s) <= end:
        w = df[
            (df["timestamp"] >= t) &
            (df["timestamp"] <  t + pd.Timedelta(seconds=window_size_s))
        ]
        if not w.empty:
            windows.append(w)
        t += pd.Timedelta(seconds=step_size_s)

    return windows

In [17]:
def normalize_rssi_per_window(window_df):
    rssi = window_df["RSSI"].values
    if len(rssi) < 2:
        window_df["rssi_norm"] = 0.0
        return window_df

    window_df["rssi_norm"] = (
        rssi - np.mean(rssi)
    ) / (np.std(rssi) + 1e-6)

    return window_df

In [ ]:
def extract_rank_based_features(window_df, top_k=3):
    features = {}

    # ---- Aggregate RSSI per beacon ----
    beacon_means = (
        window_df
        .groupby("mac_address")["RSSI"]
        .mean()
        .sort_values(ascending=False)
    )

    beacon_counts = window_df["mac_address"].value_counts()

    beacons = beacon_means.index.tolist()
    rssi_vals = beacon_means.values

    # ---- Presence ----
    features["num_beacons_seen"] = len(beacons)

    # ---- Top-K beacon IDs ----
    for i in range(top_k):
        if i < len(beacons):
            features[f"rank_{i+1}_beacon"] = beacons[i]
        else:
            features[f"rank_{i+1}_beacon"] = "NONE"

    # ---- Rank gaps ----
    if len(rssi_vals) > 1:
        features["gap_1_2"] = rssi_vals[0] - rssi_vals[1]
    else:
        features["gap_1_2"] = 0.0

    if len(rssi_vals) > 2:
        features["gap_1_3"] = rssi_vals[0] - rssi_vals[2]
    else:
        features["gap_1_3"] = 0.0

    # ---- RSSI stability (strongest beacon) ----
    strongest = beacons[0]
    rssi_strong = window_df[window_df["mac_address"] == strongest]["RSSI"]

    # ---- Packet rate ----
    features["packets_strongest"] = beacon_counts.get(strongest, 0)

    # ---- RSSI entropy ----
    if len(rssi_vals) > 1:
        probs = np.abs(rssi_vals)
        probs = probs / probs.sum()
        features["rssi_entropy"] = entropy(probs)
    else:
        features["rssi_entropy"] = 0.0

    return features

In [19]:
def extract_dataset_features(
    df,
    window_size_s=10.0,
    step_size_s=5.0
):
    windows = window_ble_data(df, window_size_s, step_size_s)
    rows = []

    for w in windows:
        feats = extract_rank_based_features(
            w
        )
        if feats is not None:
            rows.append(feats)

    return pd.DataFrame(rows)

In [ ]:
features_df = extract_dataset_features(
    df,
    window_size_s=10.0,
    step_size_s=5.0
)
features_df.head()

,num_beacons_seen,rank_1_beacon,rank_2_beacon,rank_3_beacon,gap_1_2,gap_1_3,rssi_std_strongest,rssi_iqr_strongest,packets_strongest,rssi_entropy
0,5,45:E4:9C:57:36:19,C9:17:55:E2:3E:0E,EE:E7:46:DC:19:6F,26.78169,29.000000,0.0,0.0,17,1.600285
1,6,45:E4:9C:57:36:19,38:25:68:27:CC:02,C9:17:55:E2:3E:0E,6.00000,26.773913,0.0,0.0,17,1.781506
2,8,D6:40:7F:0A:2E:59,45:E4:9C:57:36:19,38:25:68:27:CC:02,1.00000,3.500000,0.0,0.0,17,2.071531
3,8,D6:40:7F:0A:2E:59,45:E4:9C:57:36:19,38:25:68:27:CC:02,1.00000,6.000000,0.0,0.0,17,2.072255
4,2,38:25:68:27:CC:02,D0:AF:7A:95:6B:66,NONE,0.00000,0.000000,0.0,0.0,16,0.693147


In [14]:
features_df.to_csv('../data/test2.csv', index=False)